In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ============================================================
# PAPER 1
# BREAST CANCER MOLECULAR SUBTYPE CLASSIFICATION
# MODEL : RESNET50
# SECTION 1A
# IMPORTS + CONFIGURATION + ENVIRONMENT
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import os
import random
import time
import copy
import warnings
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd

from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

# -------------------------
# PyTorch
# -------------------------

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader
)

# -------------------------
# Mixed Precision
# -------------------------

from torch.cuda.amp import (
    autocast,
    GradScaler
)

# -------------------------
# Albumentations
# -------------------------

import albumentations as A
from albumentations.pytorch import ToTensorV2

# -------------------------
# TIMM
# -------------------------

import timm

# -------------------------
# Sklearn
# -------------------------

from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc

)

from sklearn.preprocessing import label_binarize

import huggingface_hub

print("=" * 80)
print("VERSIONS")
print("=" * 80)

print("Torch             :", torch.__version__)
print("TIMM              :", timm.__version__)
print("HF Hub            :", huggingface_hub.__version__)

# ============================================================
# CONFIGURATION
# ============================================================

CONFIG = {

    # -------------------------
    # Experiment
    # -------------------------

    "MODEL_NAME": "resnet50",

    "NUM_CLASSES": 4,

    "SEED": 42,

    # -------------------------
    # Image
    # -------------------------

    "IMAGE_SIZE": 224,

    # -------------------------
    # Training
    # -------------------------

    "BATCH_SIZE": 64,

    "EPOCHS": 20,

    "LEARNING_RATE": 1e-4,

    "WEIGHT_DECAY": 1e-4,

    "GRADIENT_CLIP": 1.0,

    # -------------------------
    # DataLoader
    # -------------------------

    "NUM_WORKERS": 4,

    "PIN_MEMORY": True,

    "PERSISTENT_WORKERS": True,

    # -------------------------
    # Early Stopping
    # -------------------------

    "PATIENCE": 5,

    # -------------------------
    # Device
    # -------------------------

    "DEVICE": "cuda"

    if torch.cuda.is_available()

    else "cpu",
    
    "RESUME_TRAINING": True
}

# ============================================================
# RANDOM SEED
# ============================================================

def seed_everything(seed):

    random.seed(seed)

    np.random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


seed_everything(CONFIG["SEED"])

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(CONFIG["DEVICE"])
torch.set_float32_matmul_precision("high")

print("=" * 80)
print("DEVICE")
print("=" * 80)

print("Using Device :", DEVICE)

if torch.cuda.is_available():

    print("GPU :", torch.cuda.get_device_name(0))

print()

# ============================================================
# ROOT DIRECTORIES
# ============================================================

ROOT = Path("/kaggle/input/datasets")

ANISHA = ROOT / "anishapanja"

COAUTHOR = ROOT / "hritishachoudhury"

WORK_DIR = Path("/kaggle/working")

WORK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SAVE_DIR = WORK_DIR / "resnet50_outputs"

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)
# ============================================================
# CHECKPOINT DATASET
# ============================================================

import shutil

CHECKPOINT_DIR = ANISHA / "breast-cancer-resnet50-checkpoints"

if CONFIG["RESUME_TRAINING"] and CHECKPOINT_DIR.exists():

    print("=" * 80)
    print("CHECKPOINT DATASET FOUND")
    print("=" * 80)

    for file in CHECKPOINT_DIR.rglob("*"):

        if file.is_file():

            shutil.copy2(

                file,

                SAVE_DIR / file.name

            )

            print(f"Copied : {file.name}")

else:

    print("=" * 80)
    print("NO CHECKPOINT DATASET FOUND")
    print("=" * 80)
import json

with open(
    SAVE_DIR / "config.json",
    "w"
) as f:

    json.dump(CONFIG, f, indent=4)

print("=" * 80)
print("WORKING DIRECTORY")
print("=" * 80)

print("Work :", WORK_DIR)

print("Save :", SAVE_DIR)

print()

print("=" * 80)
print("SECTION 1A COMPLETED")
print("=" * 80)
# ============================================================
# SECTION 1B
# MANIFEST + DATASET VERIFICATION + TRAIN/VAL/TEST
# ============================================================

# ============================================================
# MANIFEST PATHS
# ============================================================

MANIFEST_DIR = ANISHA / "bc-xai-final-manifest"

PATCH_CSV = MANIFEST_DIR / "patches_with_split_final.csv"

PATIENT_CSV = MANIFEST_DIR / "patient_summary.csv"

# ============================================================
# DATASET ROOTS
# ============================================================

DATASET_ROOTS = {

    "bc-xai-a2-2000-partial":
        ANISHA / "bc-xai-a2-2000-partial",

    "bc-xai-e2-2000":
        ANISHA / "bc-xai-e2-2000",

    "bc-xai-a8-patches-batch1":
        ANISHA / "bc-xai-a8-patches-batch1",

    "bc-xai-a8-patches-batch2":
        ANISHA / "bc-xai-a8-patches-batch2",

    "bc-xai-a8-patches-batch3":
        ANISHA / "bc-xai-a8-patches-batch3",

    "bc-xai-a8-patches-batch4":
        ANISHA / "bc-xai-a8-patches-batch4",

    "bc-xai-a8-patches-batch5":
        ANISHA / "bc-xai-a8-patches-batch5",

    "bc-xai-c8-patches-batch1":
        ANISHA / "bc-xai-c8-patches-batch1",

    "bc-xai-c8-patches-batch2":
        ANISHA / "bc-xai-c8-patches-batch2",

    "bc-xai-c8-patches-batch3":
        ANISHA / "bc-xai-c8-patches-batch3",

    "bc-xai-c8-patches-batch4":
        ANISHA / "bc-xai-c8-patches-batch4",

    "bc-xai-d8-patches":
        COAUTHOR / "bc-xai-d8-patches",

    "bc-xai-bh-patches":
        COAUTHOR / "bc-xai-bh-patches"

}

# ============================================================
# VERIFY DATASETS
# ============================================================

print("="*80)
print("VERIFY DATASETS")
print("="*80)

for name, path in DATASET_ROOTS.items():

    print(f"{name:<35} {path.exists()}")

# ============================================================
# LOAD CSV FILES
# ============================================================

patch_df = pd.read_csv(PATCH_CSV)

patient_df = pd.read_csv(PATIENT_CSV)

print("\n")
print("="*80)
print("MANIFEST")
print("="*80)

print("Patch CSV Shape   :", patch_df.shape)
print("Patient CSV Shape :", patient_df.shape)

# ============================================================
# LABEL ENCODING
# ============================================================

CLASS_NAMES = [

    "Basal",
    "Her2",
    "LumA",
    "LumB"

]

LABEL_MAP = {

    cls: idx

    for idx, cls in enumerate(CLASS_NAMES)

}

patch_df["label"] = patch_df["subtype_clean"].map(LABEL_MAP)

# ============================================================
# TRAIN / VAL / TEST
# ============================================================

train_df = patch_df[
    patch_df["split"] == "train"
].reset_index(drop=True)

val_df = patch_df[
    patch_df["split"] == "val"
].reset_index(drop=True)

test_df = patch_df[
    patch_df["split"] == "test"
].reset_index(drop=True)

print("\n")
print("="*80)
print("PATCH SPLITS")
print("="*80)

print(f"Train : {len(train_df):,}")
print(f"Val   : {len(val_df):,}")
print(f"Test  : {len(test_df):,}")

print("\n")
print("="*80)
print("PATIENT SPLITS")
print("="*80)

print(f"Train : {train_df.patient_id.nunique()}")
print(f"Val   : {val_df.patient_id.nunique()}")
print(f"Test  : {test_df.patient_id.nunique()}")

# ============================================================
# CLASS DISTRIBUTION
# ============================================================

print("\n")
print("="*80)
print("TRAIN CLASS DISTRIBUTION")
print("="*80)

print(train_df["subtype_clean"].value_counts())

# ============================================================
# PATIENT DISTRIBUTION
# ============================================================

print("\n")
print("="*80)
print("PATIENTS PER SITE")
print("="*80)

print(patient_df.groupby("site").size())

# ============================================================
# SANITY CHECKS
# ============================================================

assert train_df["patient_id"].nunique() == 240
assert val_df["patient_id"].nunique() == 51
assert test_df["patient_id"].nunique() == 52

assert patch_df["label"].isna().sum() == 0

print("\n")
print("="*80)
print("SECTION 1B COMPLETED SUCCESSFULLY")
print("="*80)
# ============================================================
# SECTION 2A
# DATASET CLASS
# ============================================================

print("\n")
print("="*80)
print("SECTION 2A")
print("BUILDING DATASET")
print("="*80)

# ============================================================
# BUILD ABSOLUTE IMAGE PATH
# ============================================================

def build_image_path(row):

    dataset_root = DATASET_ROOTS[row["dataset"]]

    return dataset_root / row["relative_path"]


# ============================================================
# ADD ABSOLUTE PATH COLUMN
# ============================================================

train_df["image_path"] = train_df.apply(
    build_image_path,
    axis=1
)

val_df["image_path"] = val_df.apply(
    build_image_path,
    axis=1
)

test_df["image_path"] = test_df.apply(
    build_image_path,
    axis=1
)
print("\nChecking image paths...")

train_missing = (~train_df["image_path"].apply(lambda x: x.exists())).sum()
val_missing = (~val_df["image_path"].apply(lambda x: x.exists())).sum()
test_missing = (~test_df["image_path"].apply(lambda x: x.exists())).sum()

print(f"Missing Train Images : {train_missing}")
print(f"Missing Val Images   : {val_missing}")
print(f"Missing Test Images  : {test_missing}")

assert train_missing == 0
assert val_missing == 0
assert test_missing == 0
# ============================================================
# VERIFY FEW PATHS
# ============================================================

print("\nExample paths:\n")

for p in train_df["image_path"].head():

    print(p)

# ============================================================
# DATASET CLASS
# ============================================================

class BreastCancerDataset(Dataset):

    def __init__(

        self,

        dataframe,

        transform=None

    ):

        self.df = dataframe.reset_index(drop=True)

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = cv2.imread(

            str(row.image_path)

        )

        if image is None:

            raise FileNotFoundError(

                row.image_path

            )

        image = cv2.cvtColor(

            image,

            cv2.COLOR_BGR2RGB

        )

        if self.transform is not None:

            image = self.transform(

                image=image

            )["image"]

        label = torch.tensor(

            row.label,

            dtype=torch.long

        )

        return {

            "image": image,

            "label": label,

            "patient_id": row.patient_id,

            "site": row.site,

            "subtype": row.subtype_clean

        }

print()

print("Dataset class created successfully.")

print("="*80)
print("SECTION 2A COMPLETED")
print("="*80)
# ============================================================
# SECTION 2B
# TRANSFORMS + DATASETS + DATALOADERS
# ============================================================

print("\n")
print("="*80)
print("SECTION 2B")
print("TRANSFORMS + DATALOADERS")
print("="*80)

# ============================================================
# TRAIN TRANSFORMS
# ============================================================

train_transform = A.Compose([

    A.Resize(
        CONFIG["IMAGE_SIZE"],
        CONFIG["IMAGE_SIZE"]
    ),

    A.HorizontalFlip(p=0.5),

    A.VerticalFlip(p=0.5),

    A.Rotate(
        limit=20,
        p=0.5
    ),

    A.RandomBrightnessContrast(
        brightness_limit=0.15,
        contrast_limit=0.15,
        p=0.5
    ),

    A.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.10,
        hue=0.05,
        p=0.3
    ),

    A.Normalize(

        mean=(0.485,0.456,0.406),

        std=(0.229,0.224,0.225)

    ),

    ToTensorV2()

])

# ============================================================
# VALIDATION / TEST TRANSFORMS
# ============================================================

valid_transform = A.Compose([

    A.Resize(
        CONFIG["IMAGE_SIZE"],
        CONFIG["IMAGE_SIZE"]
    ),

    A.Normalize(

        mean=(0.485,0.456,0.406),

        std=(0.229,0.224,0.225)

    ),

    ToTensorV2()

])

# ============================================================
# DATASETS
# ============================================================

train_dataset = BreastCancerDataset(

    dataframe=train_df,

    transform=train_transform

)

val_dataset = BreastCancerDataset(

    dataframe=val_df,

    transform=valid_transform

)

test_dataset = BreastCancerDataset(

    dataframe=test_df,

    transform=valid_transform

)

print()

print("Train Dataset :", len(train_dataset))

print("Validation Dataset :", len(val_dataset))

print("Test Dataset :", len(test_dataset))

# ============================================================
# CLASS WEIGHTS
# ============================================================

class_weights = compute_class_weight(

    class_weight="balanced",

    classes=np.unique(train_df["label"]),

    y=train_df["label"]

)

class_weights = torch.tensor(

    class_weights,

    dtype=torch.float32

).to(DEVICE)

print("\n")
print("="*80)
print("CLASS WEIGHTS")
print("="*80)

for i,w in enumerate(class_weights):

    print(

        CLASS_NAMES[i],

        ":",

        round(float(w),4)

    )

# ============================================================
# DATALOADERS
# ============================================================

generator = torch.Generator()

generator.manual_seed(CONFIG["SEED"])

train_loader = DataLoader(

    train_dataset,

    batch_size=CONFIG["BATCH_SIZE"],

    shuffle=True,

    drop_last=True,

    num_workers=CONFIG["NUM_WORKERS"],

    pin_memory=CONFIG["PIN_MEMORY"],
    generator=generator,

    persistent_workers=CONFIG["PERSISTENT_WORKERS"]

)
val_loader = DataLoader(

    val_dataset,

    batch_size=CONFIG["BATCH_SIZE"],

    shuffle=False,

    num_workers=CONFIG["NUM_WORKERS"],

    pin_memory=CONFIG["PIN_MEMORY"],

    persistent_workers=CONFIG["PERSISTENT_WORKERS"]

)

test_loader = DataLoader(

    test_dataset,

    batch_size=CONFIG["BATCH_SIZE"],

    shuffle=False,

    num_workers=CONFIG["NUM_WORKERS"],

    pin_memory=CONFIG["PIN_MEMORY"],

    persistent_workers=CONFIG["PERSISTENT_WORKERS"]

)

print("\n")
print("="*80)
print("DATALOADER SUMMARY")
print("="*80)

print("Train Batches :", len(train_loader))

print("Validation Batches :", len(val_loader))

print("Test Batches :", len(test_loader))
print()

print(f"Batch Size : {CONFIG['BATCH_SIZE']}")

print(f"Workers    : {CONFIG['NUM_WORKERS']}")

# ============================================================
# SANITY CHECK
# ============================================================

sample = train_dataset[0]

print("\n")
print("="*80)
print("SANITY CHECK")
print("="*80)

print("Image Shape :", sample["image"].shape)

print("Label :", sample["label"])

print("Patient :", sample["patient_id"])

print("Site :", sample["site"])

print("Subtype :", sample["subtype"])

print("\n")

print("="*80)
print("SECTION 2B COMPLETED")
print("="*80)
# ============================================================
# SECTION 3A
# MODEL UTILITIES
# ============================================================

print("\n")
print("="*80)
print("SECTION 3A")
print("MODEL UTILITIES")
print("="*80)

# ============================================================
# PARAMETER COUNTER
# ============================================================

def count_parameters(model):

    total = sum(

        p.numel()

        for p in model.parameters()

    )

    trainable = sum(

        p.numel()

        for p in model.parameters()

        if p.requires_grad

    )

    return total, trainable


# ============================================================
# MODEL FACTORY
# ============================================================

def build_model(model_name, num_classes):

    print("Creating model...")

    model = timm.create_model(

        model_name,

        pretrained=True,

        num_classes=num_classes

    )

    print("Model downloaded.")

    return model
print("="*80)
print("SECTION 3A COMPLETED")
print("="*80)
# ============================================================
# SECTION 3B
# BUILD MODEL
# ============================================================

print("\n")
print("="*80)
print("SECTION 3B")
print("BUILDING MODEL")
print("="*80)

model = build_model(

    CONFIG["MODEL_NAME"],

    CONFIG["NUM_CLASSES"]

)

model = model.to(DEVICE)
print("Model moved to GPU.")
print(f"Device : {next(model.parameters()).device}")
total_params, trainable_params = count_parameters(model)

print()


print("Model created successfully.")

print()

print("="*80)

print("MODEL SUMMARY")

print("="*80)

print("Architecture :", CONFIG["MODEL_NAME"])

print()

print(f"Total Parameters     : {total_params:,}")

print(f"Trainable Parameters : {trainable_params:,}")

print()

print("="*80)
print("SECTION 3B COMPLETED")
print("="*80)

# ============================================================
# SECTION 3C
# LOSS + OPTIMIZER + SCHEDULER + AMP
# ============================================================

print("\n")
print("="*80)
print("SECTION 3C")
print("="*80)

criterion = nn.CrossEntropyLoss(

    weight=class_weights

)

optimizer = optim.AdamW(

    model.parameters(),

    lr=CONFIG["LEARNING_RATE"],

    weight_decay=CONFIG["WEIGHT_DECAY"]

)

scheduler = optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=CONFIG["EPOCHS"],

    eta_min=1e-6

)
scaler = GradScaler()

print("Loss")

print(criterion)

print()

print("Optimizer")

print(optimizer)

print()

print("Scheduler")

print(scheduler)

print()

print("="*80)
print("SECTION 3C COMPLETED")
print("="*80)
# ============================================================
# SECTION 3D
# SANITY CHECK
# ============================================================

print("\n")
print("=" * 80)
print("SECTION 3D")
print("SANITY CHECK")
print("=" * 80)

# ------------------------------------------------------------
# GET ONE BATCH
# ------------------------------------------------------------

batch = next(iter(train_loader))

images = batch["image"].to(DEVICE)

labels = batch["label"].to(DEVICE)

print()

print(f"Images Shape : {images.shape}")

print(f"Labels Shape : {labels.shape}")

print(f"Image dtype  : {images.dtype}")

print(f"Label dtype  : {labels.dtype}")

# ------------------------------------------------------------
# FORWARD PASS
# ------------------------------------------------------------

model.train()

optimizer.zero_grad(set_to_none=True)

with autocast():

    outputs = model(images)

    loss = criterion(

        outputs,

        labels

    )

print()

print(f"Output Shape : {outputs.shape}")

print(f"Loss         : {loss.item():.6f}")

# ------------------------------------------------------------
# BACKWARD PASS
# ------------------------------------------------------------

scaler.scale(loss).backward()

scaler.unscale_(optimizer)

torch.nn.utils.clip_grad_norm_(

    model.parameters(),

    CONFIG["GRADIENT_CLIP"]

)

scaler.step(optimizer)

scaler.update()

optimizer.zero_grad(set_to_none=True)

print()

print("=" * 80)
print("SANITY CHECK PASSED")
print("=" * 80)
# ============================================================
# SECTION 4A
# METRIC UTILITIES
# ============================================================

print("\n")
print("=" * 80)
print("SECTION 4A")
print("METRIC UTILITIES")
print("=" * 80)

# ============================================================
# AVERAGE METER
# ============================================================

class AverageMeter:
    """
    Computes and stores the average and current value.
    """

    def __init__(self):
        self.reset()

    def reset(self):

        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0

    def update(self, value, n=1):

        self.val = float(value)
        self.sum += float(value) * n
        self.count += n

        if self.count > 0:
            self.avg = self.sum / self.count


# ============================================================
# METRIC TRACKER
# ============================================================

class MetricTracker:

    def __init__(self):

        self.reset()

    def reset(self):

        self.loss = AverageMeter()

        self.predictions = []

        self.targets = []

    def update(self, loss, outputs, labels):

        batch_size = labels.size(0)

        self.loss.update(

            loss.item(),

            batch_size

        )

        preds = torch.argmax(

            outputs,

            dim=1

        )

        self.predictions.extend(

            preds.detach().cpu().numpy()

        )

        self.targets.extend(

            labels.detach().cpu().numpy()

        )

    def compute(self):

        metrics = {

            "loss": self.loss.avg,

            "accuracy": accuracy_score(

                self.targets,

                self.predictions

            ),

            "precision": precision_score(

                self.targets,

                self.predictions,

                average="macro",

                zero_division=0

            ),

            "recall": recall_score(

                self.targets,

                self.predictions,

                average="macro",

                zero_division=0

            ),

            "f1": f1_score(

                self.targets,

                self.predictions,

                average="macro",

                zero_division=0

            )

        }

        return metrics


print()

print("AverageMeter  ✓")

print("MetricTracker ✓")



print()

print("=" * 80)
print("SECTION 4A COMPLETED")
print("=" * 80)
# ============================================================
# SECTION 4B
# EARLY STOPPING
# ============================================================

print("\n")
print("=" * 80)
print("SECTION 4B")
print("EARLY STOPPING")
print("=" * 80)


class EarlyStopping:
    """
    Stops training when validation metric stops improving.
    """

    def __init__(

        self,

        patience=7,

        mode="max",

        min_delta=0.0

    ):

        self.patience = patience

        self.mode = mode

        self.min_delta = min_delta

        self.best_score = None

        self.counter = 0

        self.should_stop = False

    def reset(self):

        self.best_score = None

        self.counter = 0

        self.should_stop = False

    def __call__(self, score):

        # ------------------------------------
        # First Epoch
        # ------------------------------------

        if self.best_score is None:

            self.best_score = score

            return False

        # ------------------------------------
        # Check Improvement
        # ------------------------------------

        if self.mode == "max":

            improved = score > (

                self.best_score + self.min_delta

            )

        elif self.mode == "min":

            improved = score < (

                self.best_score - self.min_delta

            )

        else:

            raise ValueError(

                "mode must be 'max' or 'min'"

            )

        # ------------------------------------
        # Improved
        # ------------------------------------

        if improved:

            self.best_score = score

            self.counter = 0

            return False

        # ------------------------------------
        # Not Improved
        # ------------------------------------

        self.counter += 1

        print(

            f"EarlyStopping "

            f"{self.counter}/{self.patience}"

        )

        if self.counter >= self.patience:

            self.should_stop = True

            return True

        return False


# ============================================================
# INITIALIZE
# ============================================================

early_stopping = EarlyStopping(

    patience=CONFIG["PATIENCE"],

    mode="max",

    min_delta=0.0

)

print()

print("EarlyStopping ✓")

print()

print("=" * 80)
print("SECTION 4B COMPLETED")
print("=" * 80)
# ============================================================
# SECTION 4C
# CHECKPOINT MANAGER
# ============================================================

print("\n")
print("=" * 80)
print("SECTION 4C")
print("CHECKPOINT MANAGER")
print("=" * 80)

BEST_MODEL_PATH = SAVE_DIR / "best_model.pth"

LAST_MODEL_PATH = SAVE_DIR / "last_model.pth"

HISTORY_PATH = SAVE_DIR / "history.pkl"

# ============================================================
# SAVE CHECKPOINT
# ============================================================

def save_checkpoint(

    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    best_val_f1,
    history,
    filename

):

    checkpoint = {

        "epoch": epoch,

        "model_state_dict": model.state_dict(),

        "optimizer_state_dict": optimizer.state_dict(),

        "scheduler_state_dict": scheduler.state_dict(),

        "scaler_state_dict": scaler.state_dict(),

        "best_val_f1": best_val_f1,

        "history": history

    }

    torch.save(

        checkpoint,

        filename

    )

# ============================================================
# LOAD CHECKPOINT
# ============================================================

def load_checkpoint(

    filename,
    model,
    optimizer=None,
    scheduler=None,
    scaler=None

):

    checkpoint = torch.load(

        filename,

        map_location=DEVICE,

        weights_only=False

    )

    model.load_state_dict(

        checkpoint["model_state_dict"]

    )

    if optimizer is not None:

        optimizer.load_state_dict(

            checkpoint["optimizer_state_dict"]

        )

    if scheduler is not None:

        scheduler.load_state_dict(

            checkpoint["scheduler_state_dict"]

        )

    if scaler is not None:

        scaler.load_state_dict(

            checkpoint["scaler_state_dict"]

        )

    epoch = checkpoint["epoch"]

    best_val_f1 = checkpoint["best_val_f1"]

    history = checkpoint["history"]

    return (

        epoch,

        best_val_f1,

        history

    )

print()

print("Best Model Path :")

print(BEST_MODEL_PATH)

print()

print("Last Model Path :")

print(LAST_MODEL_PATH)

print()

print("=" * 80)

print("SECTION 4C COMPLETED")

print("=" * 80)
# ============================================================
# SECTION 4D
# HISTORY MANAGER
# ============================================================

print("\n")
print("=" * 80)
print("SECTION 4D")
print("HISTORY MANAGER")
print("=" * 80)

# ============================================================
# INITIALIZE HISTORY
# ============================================================

if "history" not in globals():

    history = {

        "train_loss": [],

        "val_loss": [],

        "train_accuracy": [],

        "val_accuracy": [],

        "train_precision": [],

        "val_precision": [],

        "train_recall": [],

        "val_recall": [],

        "train_f1": [],

        "val_f1": [],

        "learning_rate": [],

        "epoch_time": []

    }

    print("New training history initialized.")

else:

    print("Existing training history restored.")

# ============================================================
# SAVE HISTORY CSV
# ============================================================

def save_history(

    history,
    save_dir

):

    history_df = pd.DataFrame(history)

    history_df.to_csv(

        save_dir / "history.csv",

        index=False

    )

    return history_df

# ============================================================
# SAVE TRAINING SUMMARY
# ============================================================

def save_training_summary(

    config,
    best_epoch,
    best_val_f1,
    total_time,
    history,
    save_dir

):

    with open(

        save_dir / "training_summary.txt",

        "w"

    ) as f:

        f.write("=" * 70 + "\n")

        f.write("TRAINING SUMMARY\n")

        f.write("=" * 70 + "\n\n")

        f.write(f"Model               : {config['MODEL_NAME']}\n")

        f.write(f"Epochs Completed    : {len(history['train_loss'])}\n")

        f.write(f"Best Epoch          : {best_epoch}\n")

        f.write(f"Best Validation F1  : {best_val_f1:.6f}\n")

        f.write(f"Training Time (sec) : {total_time:.2f}\n")

        f.write(f"Training Time (min) : {total_time/60:.2f}\n")

        f.write(f"Training Time (hr)  : {total_time/3600:.2f}\n")

# ============================================================
# PRINT BEST RESULT
# ============================================================

def print_best_results(

    best_epoch,
    best_val_f1

):

    print()

    print("=" * 80)

    print("BEST MODEL")

    print("=" * 80)

    print(f"Best Epoch : {best_epoch}")

    print(f"Best Validation F1 : {best_val_f1:.4f}")

# ============================================================
# READY
# ============================================================

print()

print("History Manager        ✓")

print("History CSV Export     ✓")

print("Training Summary       ✓")

print("Best Result Printer    ✓")

print()

print("=" * 80)
print("SECTION 4D COMPLETED")
print("=" * 80)
# ============================================================
# SECTION 5A
# TRAIN ONE EPOCH
# ============================================================

print("\n")
print("=" * 80)
print("SECTION 5A")
print("TRAIN ONE EPOCH")
print("=" * 80)


def train_one_epoch(

    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device

):

    model.train()

    tracker = MetricTracker()

    progress_bar = tqdm(

        loader,

        total=len(loader),

        desc="Training",

        leave=False

    )

    for batch in progress_bar:

        images = batch["image"].to(

            device,

            non_blocking=True

        )

        labels = batch["label"].to(

            device,

            non_blocking=True

        )

        optimizer.zero_grad(

            set_to_none=True

        )

        with autocast():

            outputs = model(

                images

            )

            loss = criterion(

                outputs,

                labels

            )

        scaler.scale(

            loss

        ).backward()

        scaler.unscale_(

            optimizer

        )

        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            CONFIG["GRADIENT_CLIP"]

        )

        scaler.step(

            optimizer

        )

        scaler.update()

        tracker.update(

            loss,

            outputs,

            labels

        )

        progress_bar.set_postfix(

            {

                "Loss":

                    f"{tracker.loss.avg:.4f}",

                "F1":

                    f"{f1_score(tracker.targets, tracker.predictions, average='macro', zero_division=0):.4f}"

            }

        )

    metrics = tracker.compute()

    return metrics


print()

print("train_one_epoch() created successfully.")

print()

print("=" * 80)
print("SECTION 5A COMPLETED")
print("=" * 80)
# ============================================================
# SECTION 5B
# VALIDATE ONE EPOCH
# ============================================================

print("\n")
print("=" * 80)
print("SECTION 5B")
print("VALIDATE ONE EPOCH")
print("=" * 80)


@torch.no_grad()

def validate_one_epoch(

    model,
    loader,
    criterion,
    device

):

    model.eval()

    tracker = MetricTracker()

    probabilities = []

    patient_ids = []

    sites = []

    subtypes = []

    progress_bar = tqdm(

        loader,

        total=len(loader),

        desc="Validation",

        leave=False

    )

    for batch in progress_bar:

        images = batch["image"].to(

            device,

            non_blocking=True

        )

        labels = batch["label"].to(

            device,

            non_blocking=True

        )

        with autocast():

            outputs = model(

                images

            )

            loss = criterion(

                outputs,

                labels

            )

        tracker.update(

            loss,

            outputs,

            labels

        )

        probs = torch.softmax(

            outputs,

            dim=1

        )

        probabilities.extend(

            probs.detach().cpu().numpy()

        )

        patient_ids.extend(

            batch["patient_id"]

        )

        sites.extend(

            batch["site"]

        )

        subtypes.extend(

            batch["subtype"]

        )

        progress_bar.set_postfix(

            {

                "Loss":

                    f"{tracker.loss.avg:.4f}"

            }

        )

    metrics = tracker.compute()

    metrics["probabilities"] = np.array(

        probabilities

    )

    metrics["predictions"] = np.array(

        tracker.predictions

    )

    metrics["targets"] = np.array(

        tracker.targets

    )

    metrics["patient_ids"] = patient_ids

    metrics["sites"] = sites

    metrics["subtypes"] = subtypes

    return metrics


print()

print("validate_one_epoch() created successfully.")

print()

print("=" * 80)
print("SECTION 5B COMPLETED")
print("=" * 80)
# ============================================================
# SECTION 5C1
# MAIN TRAINING LOOP (RESUMABLE)
# ============================================================

print("\n")
print("=" * 80)
print("SECTION 5C1")
print("MAIN TRAINING LOOP")
print("=" * 80)

# ============================================================
# INITIALIZATION
# ============================================================

overall_start_time = time.time()

start_epoch = 0

best_epoch = 0

best_val_f1 = -float("inf")

# ============================================================
# RESUME TRAINING IF CHECKPOINT EXISTS
# ============================================================

if CONFIG["RESUME_TRAINING"] and LAST_MODEL_PATH.exists():

    print("\n")
    print("=" * 80)
    print("RESUME TRAINING")
    print("=" * 80)

    (
        last_epoch,
        best_val_f1,
        history

    ) = load_checkpoint(

        filename=LAST_MODEL_PATH,

        model=model,

        optimizer=optimizer,

        scheduler=scheduler,

        scaler=scaler

    )

    start_epoch = last_epoch + 1

    best_epoch = last_epoch + 1

    print(f"Checkpoint Loaded : {LAST_MODEL_PATH}")

    print(f"Resuming From Epoch : {start_epoch + 1}")

    print(f"Best Validation F1 : {best_val_f1:.4f}")

    print("=" * 80)

else:

    print("\n")
    print("=" * 80)
    print("NEW TRAINING")
    print("=" * 80)

# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(start_epoch, CONFIG["EPOCHS"]):

    print("\n")
    print("=" * 80)
    print(f"Epoch {epoch + 1}/{CONFIG['EPOCHS']}")
    print("=" * 80)

    epoch_start_time = time.time()

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    train_metrics = train_one_epoch(

        model=model,

        loader=train_loader,

        criterion=criterion,

        optimizer=optimizer,

        scaler=scaler,

        device=DEVICE

    )

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    val_metrics = validate_one_epoch(

        model=model,

        loader=val_loader,

        criterion=criterion,

        device=DEVICE

    )

    # --------------------------------------------------------
    # TIME
    # --------------------------------------------------------

    epoch_time = time.time() - epoch_start_time

    current_lr = optimizer.param_groups[0]["lr"]

    # --------------------------------------------------------
    # STORE HISTORY
    # --------------------------------------------------------

    history["train_loss"].append(train_metrics["loss"])
    history["val_loss"].append(val_metrics["loss"])

    history["train_accuracy"].append(train_metrics["accuracy"])
    history["val_accuracy"].append(val_metrics["accuracy"])

    history["train_precision"].append(train_metrics["precision"])
    history["val_precision"].append(val_metrics["precision"])

    history["train_recall"].append(train_metrics["recall"])
    history["val_recall"].append(val_metrics["recall"])

    history["train_f1"].append(train_metrics["f1"])
    history["val_f1"].append(val_metrics["f1"])

    history["learning_rate"].append(current_lr)

    history["epoch_time"].append(epoch_time)

    # --------------------------------------------------------
    # PRINT RESULTS
    # --------------------------------------------------------

    print()

    print(f"Train Loss      : {train_metrics['loss']:.4f}")
    print(f"Train Accuracy  : {train_metrics['accuracy']:.4f}")
    print(f"Train Precision : {train_metrics['precision']:.4f}")
    print(f"Train Recall    : {train_metrics['recall']:.4f}")
    print(f"Train F1        : {train_metrics['f1']:.4f}")

    print()

    print(f"Val Loss        : {val_metrics['loss']:.4f}")
    print(f"Val Accuracy    : {val_metrics['accuracy']:.4f}")
    print(f"Val Precision   : {val_metrics['precision']:.4f}")
    print(f"Val Recall      : {val_metrics['recall']:.4f}")
    print(f"Val F1          : {val_metrics['f1']:.4f}")

    print()

    print(f"Learning Rate   : {current_lr:.8f}")
    print(f"Epoch Time      : {epoch_time:.2f} sec")
            # --------------------------------------------------------
    # STEP LR SCHEDULER
    # --------------------------------------------------------

    scheduler.step()

    # --------------------------------------------------------
    # BEST MODEL
    # --------------------------------------------------------

    if val_metrics["f1"] > best_val_f1:

        best_val_f1 = val_metrics["f1"]

        best_epoch = epoch + 1

        print()

        print("=" * 80)

        print("NEW BEST MODEL")

        print("=" * 80)

        print(f"Best Epoch : {best_epoch}")

        print(f"Best Validation F1 : {best_val_f1:.4f}")

        save_checkpoint(

            model=model,

            optimizer=optimizer,

            scheduler=scheduler,

            scaler=scaler,

            epoch=epoch,

            best_val_f1=best_val_f1,

            history=history,

            filename=BEST_MODEL_PATH

        )

        print("✓ Best model checkpoint saved.")

    # --------------------------------------------------------
    # SAVE LAST CHECKPOINT
    # --------------------------------------------------------

    save_checkpoint(

        model=model,

        optimizer=optimizer,

        scheduler=scheduler,

        scaler=scaler,

        epoch=epoch,

        best_val_f1=best_val_f1,

        history=history,

        filename=LAST_MODEL_PATH

    )

    # --------------------------------------------------------
    # SAVE HISTORY
    # --------------------------------------------------------

    save_history(

        history,

        SAVE_DIR

    )

    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    early_stopping(

        val_metrics["f1"]

    )

    if early_stopping.should_stop:

        print()

        print("=" * 80)

        print("EARLY STOPPING TRIGGERED")

        print("=" * 80)

        break

# ============================================================
# TRAINING FINISHED
# ============================================================

total_training_time = time.time() - overall_start_time

print()

print("=" * 80)

print("TRAINING FINISHED")

print("=" * 80)

print(f"Best Epoch : {best_epoch}")

print(f"Best Validation F1 : {best_val_f1:.4f}")

print(f"Training Time : {total_training_time/3600:.2f} hours")

print("=" * 80)
# ============================================================
# SECTION 5C3
# TRAINING FINALIZATION
# ============================================================

print("\n")
print("=" * 80)
print("SECTION 5C3")
print("TRAINING FINALIZATION")
print("=" * 80)

# ============================================================
# TOTAL TRAINING TIME
# ============================================================

total_training_time = time.time() - overall_start_time

print()

print(f"Total Training Time : {total_training_time:.2f} seconds")
print(f"                     {total_training_time/60:.2f} minutes")
print(f"                     {total_training_time/3600:.2f} hours")

# ============================================================
# SAVE FINAL HISTORY
# ============================================================

history_df = save_history(

    history=history,

    save_dir=SAVE_DIR

)

# ============================================================
# SAVE TRAINING SUMMARY
# ============================================================

save_training_summary(

    config=CONFIG,

    best_epoch=best_epoch,

    best_val_f1=best_val_f1,

    total_time=total_training_time,

    history=history,

    save_dir=SAVE_DIR

)

# ============================================================
# PRINT BEST RESULT
# ============================================================

print_best_results(

    best_epoch,

    best_val_f1

)

# ============================================================
# DISPLAY HISTORY
# ============================================================

print()

print("=" * 80)
print("TRAINING HISTORY")
print("=" * 80)

display(history_df.tail())

print()

print("History Saved :")
print(SAVE_DIR / "history.csv")

print()

print("Training Summary Saved :")
print(SAVE_DIR / "training_summary.txt")

# ============================================================
# RESTORE BEST MODEL
# ============================================================

print()

print("=" * 80)
print("LOADING BEST MODEL")
print("=" * 80)

_, _, _ = load_checkpoint(

    filename=BEST_MODEL_PATH,

    model=model,

    optimizer=optimizer,

    scheduler=scheduler,

    scaler=scaler

)

print()

print("✓ Best model restored successfully.")

print()

print("=" * 80)
print("SECTION 5C3 COMPLETED")
print("=" * 80)
# ============================================================
# SECTION 6A
# FINAL TEST EVALUATION
# ============================================================

print("\n")
print("=" * 80)
print("SECTION 6A")
print("FINAL TEST EVALUATION")
print("=" * 80)
# ============================================================
# LOAD BEST MODEL
# ============================================================

print()

print("Loading best model for testing...")

_, _, _ = load_checkpoint(

    filename=BEST_MODEL_PATH,

    model=model,

    optimizer=optimizer,

    scheduler=scheduler,

    scaler=scaler

)

model.eval()

print("Best model loaded.")
# ============================================================
# RUN TEST SET
# ============================================================

test_metrics = validate_one_epoch(

    model=model,

    loader=test_loader,

    criterion=criterion,

    device=DEVICE

)

print()

print("=" * 80)
print("TEST PERFORMANCE")
print("=" * 80)

print(f"Test Loss      : {test_metrics['loss']:.4f}")

print(f"Test Accuracy  : {test_metrics['accuracy']:.4f}")

print(f"Test Precision : {test_metrics['precision']:.4f}")

print(f"Test Recall    : {test_metrics['recall']:.4f}")

print(f"Test F1 Score  : {test_metrics['f1']:.4f}")

print()

print("=" * 80)

# ============================================================
# STORE RESULTS
# ============================================================

test_results = pd.DataFrame({

    "patient_id": test_metrics["patient_ids"],

    "site": test_metrics["sites"],

    "true_subtype": test_metrics["subtypes"],

    "true_label": test_metrics["targets"],

    "predicted_label": test_metrics["predictions"]

})

# ============================================================
# PREDICTED CLASS NAME
# ============================================================

inverse_label_map = {

    v: k

    for k, v in LABEL_MAP.items()

}

test_results["predicted_subtype"] = (

    test_results["predicted_label"]

    .map(inverse_label_map)

)

# ============================================================
# SAVE CSV
# ============================================================

prediction_csv = (

    SAVE_DIR

    / "test_predictions.csv"

)

test_results.to_csv(

    prediction_csv,

    index=False

)
print()

display(test_results.head())

print()

print(

    f"Predictions saved to:\n{prediction_csv}"

)

print()

print("=" * 80)
print("SECTION 6A COMPLETED")
print("=" * 80)
# ============================================================
# SECTION 6B
# SAVE PREDICTION PROBABILITIES
# ============================================================

print("\n")
print("=" * 80)
print("SECTION 6B")
print("SAVE PREDICTION PROBABILITIES")
print("=" * 80)

# ============================================================
# EXTRACT PROBABILITIES
# ============================================================

probabilities = test_metrics["probabilities"]

probability_df = pd.DataFrame(

    probabilities,

    columns=[

        "Probability_Basal",

        "Probability_Her2",

        "Probability_LumA",

        "Probability_LumB"

    ]

)

# ============================================================
# COMBINE RESULTS
# ============================================================

full_results = pd.concat(

    [

        test_results.reset_index(drop=True),

        probability_df.reset_index(drop=True)

    ],

    axis=1

)

# ============================================================
# PREDICTION CONFIDENCE
# ============================================================

full_results["Confidence"] = (

    probability_df.max(axis=1)

)

# ============================================================
# CORRECT / INCORRECT
# ============================================================

full_results["Correct"] = (

    full_results["true_label"]

    ==

    full_results["predicted_label"]

)

# ============================================================
# SAVE CSV
# ============================================================

full_prediction_path = (

    SAVE_DIR

    / "test_predictions_with_probabilities.csv"

)

full_results.to_csv(

    full_prediction_path,

    index=False

)

print()

print("Prediction probabilities saved.")

print(full_prediction_path)

print()

print("=" * 80)
print("CONFIDENCE SUMMARY")
print("=" * 80)

print(

    full_results["Confidence"]

    .describe()

)

print()

print("Correct Predictions :",

      full_results["Correct"].sum())

print("Incorrect Predictions :",

      (~full_results["Correct"]).sum())

print()

print("=" * 80)
print("SECTION 6B COMPLETED")
print("=" * 80)
# ============================================================
# SECTION 7A
# CONFUSION MATRIX + CLASSIFICATION REPORT
# ============================================================

print("\n")
print("=" * 80)
print("SECTION 7A")
print("CONFUSION MATRIX")
print("=" * 80)

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(

    test_metrics["targets"],

    test_metrics["predictions"]

)

cm_df = pd.DataFrame(

    cm,

    index=CLASS_NAMES,

    columns=CLASS_NAMES

)

print()

print("Confusion Matrix")

display(cm_df)

# ============================================================
# SAVE CONFUSION MATRIX CSV
# ============================================================

cm_df.to_csv(

    SAVE_DIR / "confusion_matrix.csv"

)

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

report = classification_report(

    test_metrics["targets"],

    test_metrics["predictions"],

    target_names=CLASS_NAMES,

    digits=4,

    output_dict=True,

    zero_division=0

)

report_df = pd.DataFrame(report).transpose()

print()

print("=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)

display(report_df)

# ============================================================
# SAVE REPORT CSV
# ============================================================

report_df.to_csv(

    SAVE_DIR / "classification_report.csv"

)

# ============================================================
# SAVE TXT REPORT
# ============================================================

text_report = classification_report(

    test_metrics["targets"],

    test_metrics["predictions"],

    target_names=CLASS_NAMES,

    digits=4,

    zero_division=0

)

with open(

    SAVE_DIR / "classification_report.txt",

    "w"

) as f:

    f.write(text_report)

print()

print("Classification report saved.")

print()

print("=" * 80)
print("PER-CLASS F1 SCORES")
print("=" * 80)

for cls in CLASS_NAMES:

    print(

        f"{cls:<10}: "

        f"{report_df.loc[cls,'f1-score']:.4f}"

    )

print()

print("=" * 80)
print("SECTION 7A COMPLETED")
print("=" * 80)
# ============================================================
# SECTION 7B
# CONFUSION MATRIX FIGURE
# ============================================================

print("\n")
print("=" * 80)
print("SECTION 7B")
print("CONFUSION MATRIX FIGURE")
print("=" * 80)

# ============================================================
# CREATE FIGURE
# ============================================================

fig, ax = plt.subplots(

    figsize=(8, 8)

)

im = ax.imshow(

    cm,

    interpolation="nearest"

)

plt.colorbar(im)

# ============================================================
# AXES
# ============================================================

ax.set(

    xticks=np.arange(len(CLASS_NAMES)),

    yticks=np.arange(len(CLASS_NAMES)),

    xticklabels=CLASS_NAMES,

    yticklabels=CLASS_NAMES,

    xlabel="Predicted Label",

    ylabel="True Label",

    title="Confusion Matrix"

)

plt.setp(

    ax.get_xticklabels(),

    rotation=45,

    ha="right"

)

# ============================================================
# WRITE VALUES
# ============================================================

threshold = cm.max() / 2

for i in range(cm.shape[0]):

    for j in range(cm.shape[1]):

        ax.text(

            j,

            i,

            format(cm[i, j], "d"),

            ha="center",

            va="center",

            color="white"

            if cm[i, j] > threshold

            else "black",

            fontsize=12,

            fontweight="bold"

        )

# ============================================================
# LAYOUT
# ============================================================

fig.tight_layout()

# ============================================================
# SAVE FIGURE
# ============================================================

confusion_matrix_path = (

    SAVE_DIR

    / "confusion_matrix.png"

)

plt.savefig(

    confusion_matrix_path,

    dpi=300,

    bbox_inches="tight"

)

plt.show()

print()

print("Confusion Matrix saved.")

print(confusion_matrix_path)

print()

print("=" * 80)
print("SECTION 7B COMPLETED")
print("=" * 80)
# ============================================================
# SECTION 7C
# ROC CURVES + AUC
# ============================================================

print("\n")
print("=" * 80)
print("SECTION 7C")
print("ROC CURVES")
print("=" * 80)

# ============================================================
# BINARIZE LABELS
# ============================================================

y_true = label_binarize(

    test_metrics["targets"],

    classes=np.arange(CONFIG["NUM_CLASSES"])

)

y_score = test_metrics["probabilities"]

# ============================================================
# ROC COMPUTATION
# ============================================================

fpr = {}
tpr = {}
roc_auc = {}

for i in range(CONFIG["NUM_CLASSES"]):

    fpr[i], tpr[i], _ = roc_curve(

        y_true[:, i],

        y_score[:, i]

    )

    roc_auc[i] = auc(

        fpr[i],

        tpr[i]

    )

# ============================================================
# MICRO AVERAGE
# ============================================================

fpr["micro"], tpr["micro"], _ = roc_curve(

    y_true.ravel(),

    y_score.ravel()

)

roc_auc["micro"] = auc(

    fpr["micro"],

    tpr["micro"]

)

# ============================================================
# MACRO AVERAGE
# ============================================================

all_fpr = np.unique(

    np.concatenate(

        [fpr[i] for i in range(CONFIG["NUM_CLASSES"])]

    )

)

mean_tpr = np.zeros_like(all_fpr)

for i in range(CONFIG["NUM_CLASSES"]):

    mean_tpr += np.interp(

        all_fpr,

        fpr[i],

        tpr[i]

    )

mean_tpr /= CONFIG["NUM_CLASSES"]

fpr["macro"] = all_fpr

tpr["macro"] = mean_tpr

roc_auc["macro"] = auc(

    fpr["macro"],

    tpr["macro"]

)

# ============================================================
# PLOT
# ============================================================

plt.figure(figsize=(9,8))

for i, cls in enumerate(CLASS_NAMES):

    plt.plot(

        fpr[i],

        tpr[i],

        linewidth=2,

        label=f"{cls} (AUC = {roc_auc[i]:.4f})"

    )

plt.plot(

    fpr["micro"],

    tpr["micro"],

    linestyle="--",

    linewidth=2,

    label=f"Micro Avg (AUC = {roc_auc['micro']:.4f})"

)

plt.plot(

    fpr["macro"],

    tpr["macro"],

    linestyle=":",

    linewidth=3,

    label=f"Macro Avg (AUC = {roc_auc['macro']:.4f})"

)

plt.plot(

    [0,1],

    [0,1],

    linestyle="--"

)

plt.xlim([0,1])

plt.ylim([0,1.05])

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("Multi-Class ROC Curves")

plt.legend(loc="lower right")

plt.grid(True)

roc_path = SAVE_DIR / "roc_curves.png"

plt.savefig(

    roc_path,

    dpi=300,

    bbox_inches="tight"

)

plt.show()

# ============================================================
# SAVE AUC TABLE
# ============================================================

auc_results = pd.DataFrame({

    "Class": CLASS_NAMES + ["Micro Average","Macro Average"],

    "AUC": [

        roc_auc[0],

        roc_auc[1],

        roc_auc[2],

        roc_auc[3],

        roc_auc["micro"],

        roc_auc["macro"]

    ]

})

auc_results.to_csv(

    SAVE_DIR / "auc_scores.csv",

    index=False

)

print()

print("=" * 80)

print("AUC SCORES")

print("=" * 80)

display(auc_results)

print()

print(f"ROC figure saved to:\n{roc_path}")

print()

print("=" * 80)
print("SECTION 7C COMPLETED")
print("=" * 80)
# ============================================================
# SECTION 8A
# LEARNING CURVES
# ============================================================

print("\n")
print("=" * 80)
print("SECTION 8A")
print("LEARNING CURVES")
print("=" * 80)

history_df = pd.DataFrame(history)

# ============================================================
# LOSS CURVE
# ============================================================

plt.figure(figsize=(8,6))

plt.plot(

    history_df["train_loss"],

    linewidth=2,

    label="Train"

)

plt.plot(

    history_df["val_loss"],

    linewidth=2,

    label="Validation"

)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("Training vs Validation Loss")

plt.grid(True)

plt.legend()

loss_path = SAVE_DIR / "loss_curve.png"

plt.savefig(

    loss_path,

    dpi=300,

    bbox_inches="tight"

)

plt.show()

# ============================================================
# ACCURACY CURVE
# ============================================================

plt.figure(figsize=(8,6))

plt.plot(

    history_df["train_accuracy"],

    linewidth=2,

    label="Train"

)

plt.plot(

    history_df["val_accuracy"],

    linewidth=2,

    label="Validation"

)

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.title("Training vs Validation Accuracy")

plt.grid(True)

plt.legend()

accuracy_path = SAVE_DIR / "accuracy_curve.png"

plt.savefig(

    accuracy_path,

    dpi=300,

    bbox_inches="tight"

)

plt.show()

# ============================================================
# F1 CURVE
# ============================================================

plt.figure(figsize=(8,6))

plt.plot(

    history_df["train_f1"],

    linewidth=2,

    label="Train"

)

plt.plot(

    history_df["val_f1"],

    linewidth=2,

    label="Validation"

)

plt.xlabel("Epoch")

plt.ylabel("Macro F1")

plt.title("Training vs Validation Macro F1")

plt.grid(True)

plt.legend()

f1_path = SAVE_DIR / "f1_curve.png"

plt.savefig(

    f1_path,

    dpi=300,

    bbox_inches="tight"

)

plt.show()

print()

print("=" * 80)

print("CURVES SAVED")

print("=" * 80)

print(loss_path)

print(accuracy_path)

print(f1_path)

print()

print("=" * 80)
print("SECTION 8A COMPLETED")
print("=" * 80)
# ============================================================
# SECTION 8B
# EXPERIMENT SUMMARY
# ============================================================

print("\n")
print("=" * 80)
print("SECTION 8B")
print("EXPERIMENT SUMMARY")
print("=" * 80)

# ============================================================
# BUILD SUMMARY
# ============================================================

summary = {

    "Model": CONFIG["MODEL_NAME"],

    "Image_Size": CONFIG["IMAGE_SIZE"],

    "Batch_Size": CONFIG["BATCH_SIZE"],

    "Epochs_Trained": len(history_df),

    "Best_Epoch": best_epoch,

    "Best_Validation_F1": best_val_f1,

    "Test_Accuracy": test_metrics["accuracy"],

    "Test_Precision": test_metrics["precision"],

    "Test_Recall": test_metrics["recall"],

    "Test_F1": test_metrics["f1"],

    "Macro_AUC": roc_auc["macro"],

    "Micro_AUC": roc_auc["micro"],

    "Training_Time_Seconds": total_training_time,

    "Training_Time_Minutes": total_training_time / 60,

    "Training_Time_Hours": total_training_time / 3600

}

summary_df = pd.DataFrame(

    [summary]

)

# ============================================================
# SAVE CSV
# ============================================================

summary_csv = (

    SAVE_DIR

    / "experiment_summary.csv"

)

summary_df.to_csv(

    summary_csv,

    index=False

)

# ============================================================
# SAVE TXT
# ============================================================

summary_txt = (

    SAVE_DIR

    / "experiment_summary.txt"

)

with open(

    summary_txt,

    "w"

) as f:

    f.write("=" * 70 + "\n")

    f.write("EXPERIMENT SUMMARY\n")

    f.write("=" * 70 + "\n\n")

    for key, value in summary.items():

        f.write(

            f"{key:<30}: {value}\n"

        )

# ============================================================
# DISPLAY SUMMARY
# ============================================================

print()

print("=" * 80)

print("FINAL RESULTS")

print("=" * 80)

display(summary_df)

print()

print(f"CSV Summary : {summary_csv}")

print(f"TXT Summary : {summary_txt}")

print()

print("=" * 80)
print("SECTION 8B COMPLETED")
print("=" * 80)